In [ ]:
#   Copyright 2025 UKRI-STFC

#   Licensed under the Apache License, Version 2.0 (the "License");
#   you may not use this file except in compliance with the License.
#   You may obtain a copy of the License at

#   http://www.apache.org/licenses/LICENSE-2.0

#   Unless required by applicable law or agreed to in writing, software
#   distributed under the License is distributed on an "AS IS" BASIS,
#   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#   See the License for the specific language governing permissions and
#   limitations under the License.
#
# Authors:
# Franck Vidal (URKI-STFC)

# RXSolutionsReader CT Demo

## Data format: RX Solutions

The data is in the format used by devices made by [RX Solutions](https://www.rx-solutions.com/en). The projections are saved in TIFF files. They are flatfield corrected using 16-bit unsigned integers. Metadata is saved in two different files, an XML file that can be used with orbital geometries, and a CSV file that can be used with flexible geometries.

## CIL Version

This notebook was developed using CIL v25.0.0

## Dataset

The data is available from Zenodo: https://doi.org/10.5281/zenodo.15835893

XCT datasets of a part of a lab tensile machine for in situ stress in scanning electron microscopes (SEM) acquired with the [Dual Tube High Energy (DTHE)](https://www.rx-solutions.com/en/blog/126/dthe-technology) microCT platform developed by [RX Solutions](https://www.rx-solutions.com/en) for the [MATEIS Laboratory](https://mateis.insa-lyon.fr/en) of [INSA-Lyon](https://www.insa-lyon.fr/en/). The datasets include projections and reconstructions for the two acquisitions as follows:

|	| Aquisition #1	| Acquisition #2 |
|---|---------------|----------------|
| Tube voltage (kV)	| 160		| 160 |
| Filtration		| None		| 0.4 mm of Cu |
| Amperage (uA)		| 199		| 530 |
| Exposure (s)		| 0.167		| 0.167 |
| SDD (mm)		| 807.249		| 807.248 |
| SOD (mm)		| 306.416		| 306.414 |
| Pixel pitch of the detector (mm)		| 0.15 x 0.15		| 0.15 x 0.15 |
| Voxel size in the reconstructed volume (mm)		| 0.1138667 x 0.1138667 x 0.1138667		| 0.1138668 x 0.1138668 x 0.1138668 |

The length of the sample and the oscillations of the detector make these datasets particularly useful for testing the flexible geometry reconstruction.

We acknowledge [Taith](https://www.taith.wales/), Wales' international learning exchange programme, for two travel grants to Lyon, France.

Update this filepath to where you have saved the dataset:

In [ ]:
import os

data_path = "/home/user/Downloads/filtration_none-acquisition_data"
data_path = "/home/user/Downloads/filtration_0.4mm_Cu-acquisition_data"
data_path = "/DATA/CT/2025/INSA/sample_1mm_noload"
data_path = "/DATA/CT/2026/Laminography/plaster_2p5_2p8_regular_grid"
# data_path = "/DATA/CT/2026/Laminography/plaster_2p5_2p8_poisson"

number_of_slices_to_reconstruct = 0 # Use 0 to compute it automatically
scaling_factor = 2

if scaling_factor == 1:
    roi = None
else:
    roi = {"axis_1": [None, None, scaling_factor], "axis_2": [None, None, scaling_factor]}

CSV_file_path_1 = os.path.join(data_path, 'geometry.csv')
CSV_file_path_2 = os.path.join(data_path, 'geom.csv')

if os.path.exists(CSV_file_path_1):
    CSV_file_path = CSV_file_path_1
elif os.path.exists(CSV_file_path_2):
    CSV_file_path = CSV_file_path_2
else:
    raise IOError(CSV_file_path_1 + " and " + CSV_file_path_2 + " do not exist")

In [ ]:
import numpy as np

from cil.processors import TransmissionAbsorptionConverter
from cil.framework import ImageGeometry
from cil.plugins.astra import FBP
from cil.utilities.jupyter import islicer
from cil.io.TIFF import TIFFWriter

from cil.utilities.display import show2D, show_system_positions

from readers.RXSolutionsDataReader import RXSolutionsDataReader

## Loading data

In [ ]:
reader = RXSolutionsDataReader(CSV_file_path, roi=roi)

In [ ]:
flexible_acq_data = reader.read()

In [ ]:
flexible_acq_geom = reader.get_geometry()
show_system_positions(flexible_acq_geom)

# Pre-processing and Reconstruction

In [ ]:
# Prepare the data for Astra
flexible_acq_data.reorder(order='astra');


flexible_data_exp = TransmissionAbsorptionConverter()(flexible_acq_data)

In [ ]:
# Use the system magnification to compute the voxel size
mag = flexible_data_exp.geometry.magnification
mean_mag = np.mean(mag)
print("Mean magnification: ", mean_mag)

voxel_size_xy = flexible_data_exp.geometry.config.panel.pixel_size[0] / mean_mag
voxel_size_z = flexible_data_exp.geometry.config.panel.pixel_size[1] / mean_mag

voxel_size_xy = min(voxel_size_xy, voxel_size_z)
voxel_size_z = voxel_size_xy

# Create an image geometry
num_voxel_xy = int(np.ceil(flexible_data_exp.geometry.config.panel.num_pixels[0]))
num_voxel_z = int(np.ceil(flexible_data_exp.geometry.config.panel.num_pixels[1]))

if number_of_slices_to_reconstruct > 0:
    num_voxel_z = number_of_slices_to_reconstruct // scaling_factor

image_geometry = ImageGeometry(num_voxel_xy, num_voxel_xy, num_voxel_z, voxel_size_xy, voxel_size_xy, voxel_size_z)

print(image_geometry)

In [ ]:
# Reconstruct using FDK
# Instantiate the reconstruction algorithm
fdk = FBP(image_geometry, flexible_data_exp.geometry)
fdk.set_input(flexible_data_exp)

# Perform the actual CT reconstruction
flexible_recon = fdk.get_output()

## Save the reconstruction as a stack of TIFF files

In [ ]:
writer = TIFFWriter(flexible_recon, os.path.join(data_path, "flexible/slice"), compression='uint16')
writer.write()

## Visualise the reconstruction

In [ ]:
islicer(flexible_recon, origin='upper-left')

In [ ]:
islicer(flexible_recon, direction='horizontal_y')

In [ ]:
show2D([flexible_recon.max(axis=0),flexible_recon.max(axis=1)],title=['x-y plane','x-z plane'])

In [ ]:
slice_list=[('vertical',76),('vertical',55),('horizontal_x',155),('horizontal_y',138)]
show2D(flexible_recon,slice_list=slice_list, title="FDK reconstruction")

In [ ]:
from cil.plugins.astra import ProjectionOperator
from cil.optimisation.functions import ZeroFunction, LeastSquares
from cil.optimisation.algorithms import FISTA

# Reconstructing the data using unregularised Least Squares with FISTA

In [ ]:
Projector = ProjectionOperator(image_geometry, flexible_data_exp.geometry)
LS = LeastSquares(A=Projector, b=flexible_data_exp)
fista_LS = FISTA(initial=flexible_recon, f=LS, g=ZeroFunction(), update_objective_interval=10)

In [ ]:
fista_LS.run(100)
LS_reco = fista_LS.solution
show2D(LS_reco,slice_list=slice_list, title="LS reconstruction")

# Reconstructing the data using Total Variation regularised Least Squares with FISTA

In [ ]:
from cil.plugins.ccpi_regularisation.functions import FGP_TV

In [ ]:
alpha = 0.05
TV = FGP_TV(alpha=alpha, nonnegativity=True, device='gpu')
fista_TV = FISTA(initial=flexible_recon, f=LS, g=TV, update_objective_interval=10)

In [ ]:
fista_TV.run(100)
TV_reco = fista_TV.solution
show2D(TV_reco,slice_list=slice_list, title="TV reconstruction", fix_range=(-0.02,0.07))

# Comparing the results

In [ ]:
for sl in slice_list:
    show2D([flexible_recon, LS_reco, TV_reco],slice_list=sl, title=['FDK','LS','TV'], fix_range=(-0.02,0.07),num_cols=3)